# Experiment 16 — Revision figures (after Experiment 15)

1. **16A** reruns the sign-of-adaptation analysis on the second parameter set with the model-defined therapy (debridement, then four units of perfect plaque control) instead of the instantaneous reset: treated/naive critical challenge against recovery for α = −0.5, −0.25, 0, 0.25 (α = 0.5 is recorded as therapy failure), the instant host, the dose-dependent host, the relapse ceiling and the trajectories of Figure S13D.
2. **16B** regenerates every figure affected by Experiments 15 and 16A, in the style of Paper_Figures and Revision_Figures (180 mm width, Liberation Sans, palette of the paper): **Figures 3 and 4** in full, and **Figures S6, S11, S12 and S13** in full; **Figure S15** receives two new panels (E–F) as a separate strip to place under the existing four.

Inputs: the Experiment_14_Revision, Experiment_11 exports (downloaded from the archived repository) and the Experiment_15 export (read from Google Drive, where Experiment 15 wrote it). Every figure is saved as PDF and 600-dpi PNG with the plotted data as CSV.

Checkpoints are written to Drive as in Experiment 15: if the runtime disconnects, reconnect and Run all again.

In [ ]:
import importlib, subprocess, sys
try:
    import periomemory  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/CarloGalliUnipr/periodontal-host-memory-model.git"], check=True)

In [ ]:
import glob, io, json, os, platform, shutil, time, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import font_manager
import scipy
import periomemory as pm

FAST = False                 # True: reduced grids for a smoke test
USE_DRIVE = True
RESUME = True
DRIVE_DIR = "/content/drive/MyDrive/Biosemiotic_Perio"
EXP15_NAME = "Experiment_15_Model_Revision_EXPORT"   # folder written by Experiment 15 in DRIVE_DIR
REPO_RESULTS = "https://github.com/CarloGalliUnipr/periodontal-host-memory-model/raw/main/results/"
NAME = "Experiment_16_Revision_Figures_EXPORT" + ("_FAST" if FAST else "")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    OUT = os.path.join(DRIVE_DIR, NAME)
    BASE = DRIVE_DIR
else:
    OUT, BASE = NAME, os.environ.get("PERIO_BASE", ".")
if os.path.exists(OUT) and not RESUME:
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)
print("export folder:", os.path.abspath(OUT))

CHECKS = []
_START = [0]
def part_dir(name):
    d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True); return d
def save(df, part, fname):
    df.to_csv(os.path.join(part_dir(part), fname), index=False); return df
def load(part, fname):
    return pd.read_csv(os.path.join(OUT, part, fname))
def check(part, what, value, passed=None):
    CHECKS.append(dict(module=part, check=what, value=value, passed=passed, saved=False))
    print(f"[{part}] {what}: {value}" + ("" if passed is None else f"  -> {'OK' if passed else 'FAILED'}"))
def done(part, tag):
    return os.path.exists(os.path.join(OUT, part, f"_DONE_{tag}"))
def start_part():
    _START[0] = len(CHECKS)
def mark_done(part, tag):
    new = CHECKS[_START[0]:]
    pd.DataFrame(new, columns=["module", "check", "value", "passed"]).to_csv(
        os.path.join(part_dir(part), f"checks_{tag}.csv"), index=False)
    for c in new:
        c["saved"] = True
    open(os.path.join(part_dir(part), f"_DONE_{tag}"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"[checkpoint] {part} {tag} saved")

with open(os.path.join(OUT, "software_versions.txt"), "w") as f:
    f.write(f"python {platform.python_version()}\nnumpy {np.__version__}\nscipy {scipy.__version__}\n"
            f"pandas {pd.__version__}\nmatplotlib {matplotlib.__version__}\nperiomemory {pm.__version__}\n")

## Inputs

Experiment_14_Revision and Experiment_11 exports are downloaded from the archived repository (or taken from Drive if present). The Experiment_15 export is read from Drive; if it is not found, the cell asks for the zip to be uploaded.

In [ ]:
INPUTS = os.path.join(OUT, "_inputs"); os.makedirs(INPUTS, exist_ok=True)

def find_dir(root, marker):
    """Directory under root that contains `marker` (a file or folder name)."""
    for d, subdirs, files in os.walk(root):
        if marker in subdirs or marker in files:
            return d
    return None

def get_export(name, marker, env=None):
    if env and os.environ.get(env):
        d = find_dir(os.environ[env], marker)
        if d: return d
    for root in (os.path.join(BASE, name), os.path.join(INPUTS, name)):
        if os.path.isdir(root):
            d = find_dir(root, marker)
            if d: return d
    for z in [os.path.join(BASE, name + ".zip")] + glob.glob(os.path.join(BASE, name + "*.zip")):
        if os.path.exists(z):
            zipfile.ZipFile(z).extractall(os.path.join(INPUTS, name)); return find_dir(os.path.join(INPUTS, name), marker)
    url = REPO_RESULTS + name + ".zip"
    try:
        print("downloading", url)
        data = urllib.request.urlopen(url, timeout=120).read()
        zipfile.ZipFile(io.BytesIO(data)).extractall(os.path.join(INPUTS, name))
        return find_dir(os.path.join(INPUTS, name), marker)
    except Exception as e:
        print("download failed:", e)
    if IN_COLAB:
        from google.colab import files
        print(f"Upload {name}.zip")
        up = files.upload()
        for fn, b in up.items():
            zipfile.ZipFile(io.BytesIO(b)).extractall(os.path.join(INPUTS, name))
        return find_dir(os.path.join(INPUTS, name), marker)
    raise FileNotFoundError(name)

E14 = get_export("Experiment_14_Revision_EXPORT", "14A_rechallenge", env="PERIO_EXP14")
E11 = get_export("Experiment_11_Sign_of_Adaptation_EXPORT", "periodic_thresholds_by_sign.csv", env="PERIO_EXP11")
E15 = get_export(EXP15_NAME, "15A_intrinsic_bistability", env="PERIO_EXP15")
print("Experiment 14:", E14); print("Experiment 11:", E11); print("Experiment 15:", E15)
r14 = lambda sub, f: pd.read_csv(os.path.join(E14, sub, f))
r15 = lambda sub, f: pd.read_csv(os.path.join(E15, sub, f))
r11 = lambda f: pd.read_csv(os.path.join(E11, f))
s15 = json.load(open(os.path.join(E15, "run_settings.json")))
if s15.get("FAST"):
    print("WARNING: the Experiment 15 export is a FAST run; figures from it are provisional.")

## 16A. Sign of adaptation on the second set with the defined therapy

In [ ]:
SECOND = dict(pm.SECOND)
UB2, EPS2 = 0.25, 0.03
ALPHAS = [-0.5, -0.25, 0.0, 0.25, 0.5]
ET = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0] if not FAST else [0.0, 1.0, 4.0]
STEP = 0.005 if not FAST else 0.02
GRID = np.round(np.arange(UB2 + STEP, 4.0 + 1e-9, STEP), 4)
GRID_DOSE = np.round(np.arange(UB2 + STEP, 1.5 + 1e-9, STEP), 4)

def conv_grid(m, state, ub, t_on, dur, settle, grid):
    """For each onset in t_on, the smallest step challenge on `grid` that leaves H > 0.5 after `settle`
    (nan if none) and whether conversion is monotone in challenge size."""
    nT, nG = len(t_on), len(grid)
    st = tuple(np.repeat(np.asarray(x, float).ravel()[:1], nT * nG) for x in state)
    tt = np.repeat(pm.align(np.asarray(t_on, float)), nG); UU = np.tile(grid, nT)
    out, _ = m.run(st, pm.schedule(ub, tt, pm.align(dur), UU), 0.0, float(tt.max() + pm.align(dur) + settle))
    conv = (out[0] > 0.5).reshape(nT, nG)
    crit, mono = [], True
    for row in conv:
        if row.any():
            i = int(np.argmax(row)); crit.append(grid[i]); mono &= bool(row[i:].all())
        else:
            crit.append(np.nan)
    return np.array(crit), mono

def defined_therapy(m, p, ub, b=0.0):
    hi = pm.eq_state(p, ub, "high", gamma=0.0, b=b)
    st, _, ret = pm.therapy(m, pm.tile(hi, 1))
    after, _ = m.run(st, pm.const(ub), 0.0, 10 / m.eps.item() if not m.instant else 200.0)
    return st, bool(after[0][0] < 0.5), float(ret[0])

P = "16A_second_set_defined_therapy"
if not done(P, "window"):
    start_part()
    rows, inst = [], []
    for a in ALPHAS:
        p = dict(SECOND, a=a)
        m = pm.Model(**p, eps=EPS2)
        lo = pm.eq_state(p, UB2, "low", gamma=0.0); hi = pm.eq_state(p, UB2, "high", gamma=0.0)
        tr, restored, ret = defined_therapy(m, p, UB2)
        t_on = np.array(ET) / EPS2
        cn, mn = conv_grid(m, lo, UB2, t_on, 2 / EPS2, 10 / EPS2, GRID)
        if restored:
            ct, mt = conv_grid(m, tr, UB2, t_on, 2 / EPS2, 10 / EPS2, GRID)
        else:
            ct, mt = np.full(len(ET), np.nan), True
        cr, mr = conv_grid(m, (lo[0], lo[1], hi[2], 0.0), UB2, t_on, 2 / EPS2, 10 / EPS2, GRID)
        if a >= 0:
            check(P, f"alpha={a}: conversion monotone (naive, treated, reset)", (mn, mt, mr), bool(mn and mt and mr))
        else:   # not cooperative: non-monotone conversion is a property of the model (SI Section 12.3)
            check(P, f"alpha={a}: conversion monotone (naive, treated, reset) [informational, alpha < 0]", (mn, mt, mr))
        for k, e in enumerate(ET):
            rows.append(dict(alpha=a, eps=EPS2, eps_T_rec=e, therapy_restores=restored, A_retained=ret,
                             crit_naive=cn[k], crit_treated=ct[k], crit_reset=cr[k],
                             ratio_treated=ct[k] / cn[k], ratio_reset=cr[k] / cn[k]))
        mi = pm.Model(**p, eps=EPS2, instant=True)
        ci, _ = conv_grid(mi, lo, UB2, np.zeros(1), 20.0, 200.0, GRID)
        inst.append(dict(alpha=a, crit_instant=ci[0]))
    W2 = save(pd.DataFrame(rows), P, "window_by_alpha.csv")
    save(pd.DataFrame(inst), P, "instant_host.csv")
    z = W2[W2.eps_T_rec == 0]
    print(z[["alpha", "therapy_restores", "crit_naive", "crit_treated", "crit_reset", "ratio_treated", "ratio_reset"]].round(4))
    r0 = z.set_index("alpha")
    check(P, "alpha=0: treated/naive = 1", round(float(r0.loc[0.0, "ratio_treated"]), 4),
          abs(r0.loc[0.0, "ratio_treated"] - 1) < 0.02)
    check(P, "alpha=-0.25: treated/naive > 1 (tolerance)", round(float(r0.loc[-0.25, "ratio_treated"]), 4),
          bool(r0.loc[-0.25, "ratio_treated"] > 1))
    check(P, "alpha=0.25: treated/naive < 1 (sensitization)", round(float(r0.loc[0.25, "ratio_treated"]), 4),
          bool(r0.loc[0.25, "ratio_treated"] < 1))
    check(P, "alpha=0.5: defined therapy restores the site", bool(r0.loc[0.5, "therapy_restores"]),
          not bool(r0.loc[0.5, "therapy_restores"]))
    e11 = r11("tipping_window_by_sign.csv")
    v = e11[(e11.a == -0.25) & (e11.eps_T_rec == 0)].critical_naive.iloc[0]
    check(P, f"naive critical at alpha=-0.25 reproduces Experiment 11 ({v:.3f})", float(r0.loc[-0.25, "crit_naive"]),
          abs(r0.loc[-0.25, "crit_naive"] - v) < 2 * STEP + 0.01)
    mark_done(P, "window")
W2 = load(P, "window_by_alpha.csv")

In [ ]:
if not done(P, "dose"):
    start_part()
    p = dict(SECOND, a=1.0); b = 1.4
    m = pm.Model(**p, b=b, eps=EPS2)
    lo = pm.eq_state(p, UB2, "low", gamma=0.0, b=b)
    tr, restored, ret = defined_therapy(m, p, UB2, b=b)
    g_st, _ = m.run(pm.tile(lo, 1), pm.const(0.33), 0.0, 5 / EPS2)
    t_on = np.array(ET) / EPS2
    rows = []
    cn, mn = conv_grid(m, lo, UB2, t_on, 2 / EPS2, 10 / EPS2, GRID_DOSE)
    for hist, st in (("gingivitis", g_st), ("treated_protocol", tr)):
        c, mono = conv_grid(m, st, UB2, t_on, 2 / EPS2, 10 / EPS2, GRID_DOSE)
        check(P, f"dose-dependent host, {hist}: conversion monotone", bool(mono and mn), bool(mono and mn))
        for k, e in enumerate(ET):
            rows.append(dict(history=hist, eps_T_rec=e, crit=c[k], crit_naive=cn[k], ratio=c[k] / cn[k]))
    check(P, "dose-dependent host: defined therapy restores the site", restored, restored)
    D2 = save(pd.DataFrame(rows), P, "dose_dependent.csv")
    print(D2[D2.eps_T_rec == 0].round(4))
    mark_done(P, "dose")
D2 = load(P, "dose_dependent.csv")

In [ ]:
if not done(P, "ceiling"):
    start_part()
    rows = []
    for a in ALPHAS:
        p = dict(SECOND, a=a); f = pm.folds(p)
        ubs = np.round(np.arange(0.07, f[1] + 1e-9, 0.005 if not FAST else 0.02), 4)
        for host, inst_ in (("eps=0.03", False), ("instant", True)):
            m = pm.Model(**p, eps=EPS2, instant=inst_)
            rel = []
            for ub in ubs:
                hi = pm.eq_state(p, ub, "high", gamma=0.0)
                lo = pm.eq_state(p, ub, "low", gamma=0.0)
                if hi is None or abs(hi[0] - lo[0]) < 1e-6:
                    rel.append(np.nan); continue
                st, _, _ = pm.therapy(m, pm.tile(hi, 1))
                st, _ = m.run(st, pm.const(ub), 0.0, 200.0 if inst_ else 10 / EPS2)
                rel.append(float(st[0][0] > 0.5))
            rel = np.array(rel); ceil = np.nan
            for i in range(len(ubs)):
                if np.all(rel[i:] == 1.0):
                    ceil = ubs[i]; break
            rows.append(dict(alpha=a, host=host, relapse_ceiling=ceil, static_fold=f[1],
                             ceiling_over_fold=ceil / f[1], relapse_at_baseline_0_25=bool(rel[np.argmin(abs(ubs - UB2))] == 1)))
    C2 = save(pd.DataFrame(rows), P, "relapse_ceiling.csv"); print(C2.round(3))
    # equilibrium ranges for Figure S13A
    rows = []
    for a in np.round(np.arange(-0.5, 0.5001, 0.025), 3):
        f = pm.folds(dict(SECOND, a=a))
        rows.append(dict(alpha=a, u_up=f[1], u_down=f[3]))
    save(pd.DataFrame(rows), P, "folds_by_alpha.csv")
    rows = []
    for a in np.round(np.arange(0.0, 0.5001, 0.025), 3):
        p = dict(SECOND, a=a)
        for host, inst_ in (("eps=0.03", False), ("instant", True)):
            m = pm.Model(**p, eps=EPS2, instant=inst_)
            rows.append(dict(alpha=a, host=host, therapy_restores=defined_therapy(m, p, UB2)[1]))
    TB = save(pd.DataFrame(rows), P, "therapy_by_alpha.csv")
    fails = TB[(TB.host == "eps=0.03") & (~TB.therapy_restores)].alpha
    check(P, "second set: smallest alpha at which the defined therapy fails (eps=0.03)", float(fails.min()) if len(fails) else None)
    mark_done(P, "ceiling")
C2 = load(P, "relapse_ceiling.csv"); FOLDS2 = load(P, "folds_by_alpha.csv"); TB2 = load(P, "therapy_by_alpha.csv")
A_FAIL = float(TB2[(TB2.host == "eps=0.03") & (~TB2.therapy_restores)].alpha.min())

In [ ]:
if not done(P, "traces"):
    start_part()
    p = dict(SECOND, a=-0.5); T = 60.0; peak = 0.9; ncyc = 40
    umax = peak / (1 - np.exp(-pm.K_REGROW * T)); u_fun = lambda t: umax * (1 - np.exp(-pm.K_REGROW * t))
    rows = []
    for host, inst_ in (("eps=0.03", False), ("instant", True)):
        m = pm.Model(**p, eps=EPS2, instant=inst_)
        st = tuple(np.zeros(1) for _ in range(4))
        for c in range(ncyc):
            rec = []
            def mon(t, H, M, A, D):
                rec.append((t, H[0], A[0]))
            st, _ = m.run(st, u_fun, 0.0, T, monitor=mon)
            if c >= ncyc - 2:
                for t, H, A in rec[::5]:
                    rows.append(dict(host=host, t=(c - (ncyc - 2)) * T + t, H=H, A=A, u=float(u_fun(t))))
    save(pd.DataFrame(rows), P, "traces_alpha_m05.csv")
    mark_done(P, "traces")
TR2 = load(P, "traces_alpha_m05.csv")

## 16B. Figures

Style of Paper_Figures and Revision_Figures: 180 mm width, Liberation Sans, bold left-aligned panel titles, light grid; instant host #eb6834, treated #6250d6, resolved gingivitis #1baf7a, naive dark grey dashed, memory rates blue light → dark.

In [ ]:
if IN_COLAB:
    subprocess.run("apt-get -qq install -y fonts-liberation > /dev/null 2>&1", shell=True)
    font_manager._load_fontmanager(try_read_cache=False)
fams = {f.name for f in font_manager.fontManager.ttflist}
FONT = "Liberation Sans" if "Liberation Sans" in fams else "DejaVu Sans"
plt.rcParams.update({"font.family": FONT, "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "axes.grid": True,
                     "grid.color": "#e6e6e6", "grid.linewidth": 0.5, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.linewidth": 0.6, "lines.linewidth": 1.3,
                     "lines.markersize": 3.5, "legend.frameon": False, "pdf.fonttype": 42})
MM = 1 / 25.4; WIDTH = 180 * MM
INST, TREAT, GING, NAIVE = "#eb6834", "#6250d6", "#1baf7a", "#444444"
BLUE = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
FIGDIR = part_dir("figures")
SRC = []

def title(ax, letter, text):
    ax.set_title(f"{letter}  {text}", loc="left", fontweight="bold")

def src(fig, panel, df):
    d = os.path.join(FIGDIR, fig); os.makedirs(d, exist_ok=True)
    df.to_csv(os.path.join(d, f"{fig}_{panel}_data.csv"), index=False)

def finish(fig, name):
    fig.tight_layout(pad=0.6, h_pad=1.0, w_pad=1.0)
    d = os.path.join(FIGDIR, name); os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, f"{name}.pdf"))
    fig.savefig(os.path.join(d, f"{name}.png"), dpi=600)
    plt.close(fig); print("figure", name)

def host_color(h):
    if h == "instant": return INST
    return BLUE.get(float(h.split("=")[1]), "#2a78d6")

# shared data
RC = r14("14A_rechallenge", "rechallenge_reference.csv")
LP = r14("14A_rechallenge", "summary_lowest_persistent_U.csv")
RW = r14("14A_rechallenge", "summary_relapse_without_challenge.csv")
SS = r14("14A_rechallenge", "rechallenge_single_state.csv")
B1 = r14("14B_tipping_window", "B1_critical_vs_duration.csv")
B2 = r14("14B_tipping_window", "B2_window_vs_recovery.csv")
B2A = r14("14B_tipping_window", "B2_memory_decay_treated_protocol.csv")
B2H = r14("14B_tipping_window", "B2_half_lives_and_window_lengths.csv")
B4 = r14("14B_tipping_window", "B4_relapse_ceiling_summary.csv")
B5 = r14("14B_tipping_window", "B5_plaque_control_sensitivity.csv")
D2S = r14("14D_episodic", "D2_scan_common_draws.csv")
D3 = r14("14D_episodic", "D3_activity_matched.csv")
SEQ = np.load(os.path.join(E14, "14D_episodic", "per_site_loss_sequences.npz"))
E14S = r14("14E_robustness_sample", "E_sample_results.csv")
C15 = r15("15C_calibrated_gingivitis", "C_calibrated_rechallenge.csv")
C15U = r15("15C_calibrated_gingivitis", "C_reversibility_and_Ustar.csv")
A3W = r15("15A_intrinsic_bistability", "A3_window_vs_recovery.csv")
B15 = r15("15B_intrinsic_sample", "B_sample_results.csv")
D15 = r15("15D_matched_bleeding", "D_matched_bleeding.csv")
E15M = r15("15E_safe_challenge", "E_safe_map.csv")
F1 = r15("15F_repeated_instrumentation", "F1_single_vs_repeated.csv")
F2 = r15("15F_repeated_instrumentation", "F2_debridement_every_recall.csv")
G15 = r15("15G_weibull", "G_weibull.csv")
U_UP_REF = pm.folds(pm.REFERENCE)[1]

### Figure 3

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(WIDTH, 6.9))
ax = axs[0, 0]
c = C15[(C15.set == "reference") & (C15.r == 1.0) & (C15.challenge == "own U*")]
nv = c[(c.history == "naive") & (c.host == "eps=0.01")]
ax.plot(nv.days, nv.peak_H, "--", color=NAIVE, label="naive (each host)", zorder=6)
for h, lab in (("instant", "treated, instant host"), ("eps=0.03", "treated, ε = 0.03"), ("eps=0.01", "treated, ε = 0.01")):
    g = c[(c.history == "treated") & (c.host == h)]
    ax.plot(g.days, g.peak_H, "o-", color=host_color(h), label=lab, mfc=host_color(h))
    gp = g[g.persistent]
    ax.plot(gp.days, gp.peak_H, "o", mfc="white", mec=host_color(h), ms=5, zorder=5)
ax.set_xlabel("days since therapy"); ax.set_ylabel("peak H during challenge"); ax.set_ylim(0, 1.05)
title(ax, "A", "Calibrated rechallenge (naive peak 0.30)")
ax.legend(loc="lower center", ncol=2, handlelength=1.6)
src("Figure3", "A", c)

ax = axs[0, 1]
lp = LP[(LP.baseline == 0.2) & (LP.history == "treated_D0")]
days_all = [0, 15, 30, 60, 90, 180, 365]
for h in ("eps=0.03", "eps=0.01"):
    g = lp[lp.host == h].set_index("recovery_days").reindex(days_all)
    ax.plot(g.index, g.lowest_U_persistent, "o-", color=host_color(h), label=f"treated, ε = {h.split('=')[1]}")
    miss = g[g.lowest_U_persistent.isna()].index
    ax.plot(miss, np.full(len(miss), 0.72), "^", mfc="white", mec=host_color(h), ms=4)
ax.axhline(0.55, color=INST, lw=1.2, label="instant host (any history)")
ax.set_ylim(0.25, 0.75); ax.set_xlabel("days since therapy"); ax.set_ylabel("smallest converting plateau")
title(ax, "B", "The rechallenge itself can convert the site"); ax.legend(loc="lower right")
src("Figure3", "B", lp)

ax = axs[1, 0]
ss = SS[SS.U == 0.3]
for h in ("eps=0.1", "eps=0.03", "eps=0.01", "instant"):
    g = ss[(ss.host == h) & (ss.history == "conditioned")].sort_values("recovery_days")
    ax.plot(g.recovery_days, g.peak_H, "o-", color=host_color(h), label=("conditioned, " + (f"ε = {h.split('=')[1]}" if h != "instant" else "instant host")))
g = ss[(ss.host == "eps=0.01") & (ss.history == "naive")].sort_values("recovery_days")
ax.plot(g.recovery_days, g.peak_H, "--", color=NAIVE, label="naive, ε = 0.01")
g = ss[(ss.host == "instant") & (ss.history == "naive")].sort_values("recovery_days")
ax.plot(g.recovery_days, g.peak_H, "--", color=INST, lw=0.9, label="naive, instant host")
ax.set_ylim(0, 1.05); ax.set_xlabel("days after conditioning"); ax.set_ylabel("peak H during challenge")
title(ax, "C", "Same test, single stable state"); ax.legend(loc="upper right")
src("Figure3", "C", ss)

ax = axs[1, 1]
b = B1[B1.host == "eps=0.03"]
for hist, col, lab, ls in (("naive", NAIVE, "naive", "--"), ("gingivitis", GING, "resolved gingivitis", "-"),
                           ("treated_protocol", TREAT, "treated", "-")):
    g = b[b.history == hist].sort_values("duration")
    ax.plot(g.duration_days, g.critical, ls, marker="o", color=col, label=lab)
    miss = g[g.critical.isna()]
    ax.plot(miss.duration_days, np.full(len(miss), 0.80), "^", mfc="white", mec=col, ms=4)
gi = B1[(B1.host == "instant") & (B1.history == "naive")].sort_values("duration")
ax.plot(gi.duration_days, gi.critical, "o-", color=INST, label="instant host")
ax.axhline(U_UP_REF, ls=":", color="k", lw=0.8); ax.axhline(0.2, color="k", lw=0.8)
ax.set_xscale("log"); ax.set_ylim(0.15, 0.85); ax.set_xlabel("challenge duration (days)"); ax.set_ylabel("critical challenge")
title(ax, "D", "Critical challenge by history (ε = 0.03)"); ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncol=2)
src("Figure3", "D", pd.concat([b, gi]))

ax = axs[2, 0]
bb = B2[B2.history == "treated_protocol"]
for h in ("eps=0.1", "eps=0.03", "eps=0.01"):
    g = bb[bb.host == h].sort_values("eps_T_rec")
    ax.plot(g.eps_T_rec, g.V, "o-", color=host_color(h), label=f"ε = {h.split('=')[1]}")
ga = B2A[B2A.host == "eps=0.03"]
ax.plot(ga.eps_T, ga.A_rel, ":", color="k", label="adaptive state A (ε = 0.03)")
x = np.linspace(0, 8, 200); ax.plot(x, np.exp(-x), color="k", lw=0.8, label="passive decay")
ax.set_xlim(0, 4); ax.set_xlabel("recovery (memory times, εT)"); ax.set_ylabel("normalized vulnerability V")
title(ax, "E", "Vulnerability decays on the memory time scale"); ax.legend(loc="upper right")
src("Figure3", "E", bb)

ax = axs[2, 1]
b4 = B4[B4.kind == "treated_protocol"].copy(); b4["eps"] = [float(h.split("=")[1]) if h != "instant" else np.nan for h in b4.host]
g = b4.dropna(subset=["eps", "ceiling"]).sort_values("eps")
ax.fill_between(g.eps, g.ceiling, U_UP_REF, color=TREAT, alpha=0.15)
ax.plot(g.eps, g.ceiling, "o-", color=TREAT, label="treated sites")
ax.axhline(0.234, ls="--", color="k", lw=0.8, label="slow-memory limit (0.234)")
ax.axhline(U_UP_REF, ls=":", color="k", lw=0.8, label="static fold")
ax.set_xscale("log"); ax.set_ylim(0.2, 0.5); ax.set_xlabel("memory rate ε"); ax.set_ylabel("baseline plaque")
title(ax, "F", "Lowest baseline plaque for relapse"); ax.legend(loc="lower right")
src("Figure3", "F", b4)
finish(fig, "Figure3")

### Figure 4

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(WIDTH, 4.6))
ax = axs[0, 0]
z = W2[W2.eps_T_rec == 0]
ok = z[z.therapy_restores & z.ratio_treated.notna()]
ax.plot(ok.alpha, ok.ratio_treated, "s", color=TREAT, ms=5, label="second set (bistable at α = 0)")
nn = z[z.therapy_restores & z.crit_naive.isna()]
for a in nn.alpha:
    ax.plot(a, 1.12, "o", mfc="white", mec=TREAT, ms=5); ax.annotate("naive never\nconverts", (a, 1.12), (a + 0.03, 1.10), fontsize=5.5)
for a in z[~z.therapy_restores].alpha:
    ax.plot(a, 0.62, "x", color=TREAT, ms=5); ax.annotate("therapy\nfails", (a, 0.62), (a - 0.21, 0.585), fontsize=5.5)
for e, ls in ((0.03, "-"), (0.01, "--")):
    g = A3W[(A3W.eps == e) & (A3W.rec_memory_units == 0) & (A3W.alpha >= 0)].sort_values("alpha")
    ax.plot(g.alpha, g.ratio, ls, color=BLUE[e], label=f"third set, ε = {e}")
ax.axhline(1, color="k", lw=0.6); ax.set_ylim(0.55, 1.18); ax.set_xlabel("adaptation strength α")
ax.set_ylabel("treated / naive critical challenge"); title(ax, "A", "The sign of adaptation"); ax.legend(loc="lower left")
src("Figure4", "A", pd.concat([z.assign(set="second"), A3W[A3W.rec_memory_units == 0].assign(set="third")]))

ax = axs[0, 1]
for a, col in ((-0.25, GING), (0.25, TREAT)):
    g = W2[(W2.alpha == a)].sort_values("eps_T_rec")
    ax.plot(g.eps_T_rec, g.ratio_treated, "o-", color=col, label=f"second set, α = {a}")
g = A3W[(A3W.alpha == 0.5) & (A3W.eps == 0.03)].sort_values("rec_memory_units")
ax.plot(g.rec_memory_units, g.ratio, "s--", color=BLUE[0.03], label="third set, α = 0.5")
ax.axhline(1, color="k", lw=0.6); ax.set_xlabel("recovery (memory times, εT)"); ax.set_ylabel("treated / naive")
title(ax, "B", "Both effects fade as memory recovers"); ax.legend(loc="lower right")
src("Figure4", "B", W2)

ax = axs[0, 2]
for hist, col, lab in (("gingivitis", GING, "resolved gingivitis"), ("treated_protocol", TREAT, "treated periodontitis")):
    g = D2[D2.history == hist].sort_values("eps_T_rec")
    ax.plot(g.eps_T_rec, g.ratio, "o-", color=col, label=lab)
ax.axhline(1, color="k", lw=0.6); ax.set_xlabel("recovery (memory times, εT)"); ax.set_ylabel("treated / naive")
title(ax, "C", "Dose-dependent adaptation"); ax.legend(loc="lower right")
src("Figure4", "C", D2)

ax = axs[1, 0]
def example_sites(key, k=6):
    L = SEQ[key]; act = (L > 1e-4); fr = act.mean(1); target = fr.mean()
    idx = np.argsort(np.abs(fr - target), kind="stable")[:k]
    return act[np.sort(idx)]
y = 0; rows = []
for key, col, lab in (("matched_instant_cv0.15_target0.2", INST, "instant host"),
                      ("matched_eps=0.01_cv0.15_target0.2", BLUE[0.01], "ε = 0.01")):
    A_ = example_sites(key)
    for s in A_:
        iv = np.where(s)[0]; ax.vlines(iv + 1, y + 0.1, y + 0.9, color=col, lw=0.8)
        rows += [dict(host=lab, row=y, interval=int(i) + 1) for i in iv]; y += 1
    ax.text(152, y - 3, lab, color=col, va="center", fontsize=6)
    y += 0.6
ax.set_yticks([]); ax.set_xlim(0, 175); ax.set_xlabel("recall interval"); ax.grid(False)
title(ax, "D", "Equal activity (20%), two hosts")
src("Figure4", "D", pd.DataFrame(rows))

for ax, col_, lab, letter, ttl in ((axs[1, 1], "lag1_autocorr", "lag-1 autocorrelation", "E", "Memory clusters activity"),
                                   (axs[1, 2], "top10_share", "share of loss in worst 10%", "F", "… but does not concentrate loss")):
    for h in ("instant", "eps=0.03", "eps=0.01"):
        for cv, ls in ((0.15, "-"), (0.3, "--")):
            g = D2S[(D2S.host == h) & (D2S.cv == cv)].sort_values("active_fraction")
            ax.plot(g.active_fraction, g[col_], ls, color=host_color(h), lw=1.0)
            m_ = D3[(D3.host == h) & (D3.cv == cv)]
            ax.plot(m_.active_fraction, m_[col_], "o", color=host_color(h), ms=3)
    if col_ == "top10_share": ax.axhline(0.10, ls="--", color="k", lw=0.6)
    ax.set_xlabel("fraction of intervals with active loss"); ax.set_ylabel(lab); title(ax, letter, ttl)
axs[1, 1].legend(handles=[Line2D([], [], color=INST, label="instant host"), Line2D([], [], color=BLUE[0.03], label="ε = 0.03"),
                          Line2D([], [], color=BLUE[0.01], label="ε = 0.01"), Line2D([], [], color="k", ls="-", label="CV 0.15"),
                          Line2D([], [], color="k", ls="--", label="CV 0.30")], loc="upper left")
src("Figure4", "E_F", pd.concat([D2S, D3]))
finish(fig, "Figure4")

### Figure S6

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(WIDTH, 4.6))
e14 = E14S[E14S.ratio_treated_naive.notna()]; b15 = B15[B15.category == "evaluable"]
ref_r = float(e14[e14.reference].ratio_treated_naive.iloc[0])
third_r = float(A3W[(A3W.alpha == 0.5) & (A3W.eps == 0.03) & (A3W.rec_memory_units == 0)].ratio.iloc[0])
for ax, col, letter, ttl, xl in ((axs[0, 0], "ratio_treated_naive", "A", "Treated / naive in both samples", "treated / naive critical challenge"),
                                 (axs[0, 1], "ratio_treated_instant", "B", "Treated / instant host", "treated / instant critical challenge")):
    bins = np.linspace(0.25, 1.0, 31)
    ax.hist(e14[col].dropna(), bins=bins, color="#9aa9c9", alpha=0.85, label=f"reference neighbourhood ({e14[col].notna().sum()})")
    ax.hist(b15[col].dropna(), bins=bins, color=TREAT, alpha=0.6, label=f"intrinsically bistable ({b15[col].notna().sum()})")
    ax.axvline(1, color="k", lw=0.6)
    if col == "ratio_treated_naive":
        ax.axvline(ref_r, ls="--", color="#4a5a80", lw=0.8); ax.axvline(third_r, ls="--", color=TREAT, lw=0.8)
    ax.set_xlabel(xl); ax.set_ylabel("parameter sets"); title(ax, letter, ttl); ax.legend(loc="upper left")
src("FigureS6", "A_B", pd.concat([e14.assign(sample="reference neighbourhood"), b15.assign(sample="third-set neighbourhood")]))
ax = axs[0, 2]
e = E14S[E14S.ratchet_margin_over_instant.notna()]
ax.scatter(e.u_down / e.u_up, e.ratchet_margin_over_instant, s=5, color="#5598e7", alpha=0.7)
rf = E14S[E14S.reference]; ax.plot(rf.u_down / rf.u_up, rf.ratchet_margin_over_instant, "*", color="k", ms=8)
ax.axhline(1, color="k", lw=0.6); ax.set_xlabel("u↓ / u↑"); ax.set_ylabel("tolerated peak, ε = 0.01 / instant")
title(ax, "C", "Ratchet margin and folds"); src("FigureS6", "C", e)
ax = axs[1, 0]
s1 = E14S[E14S.stage1_exists]
ax.hist(s1.D_irr, bins=25, color="#5598e7"); ax.axvline(float(rf.D_irr.iloc[0]), ls="--", color="k", lw=0.8)
ax.text(0.97, 0.95, f"no hygiene-only stage:\n{(~E14S.stage1_exists).sum()} of {len(E14S)} sets", transform=ax.transAxes, ha="right", va="top", fontsize=6)
ax.set_xlabel("hygiene limit D_irr"); ax.set_ylabel("parameter sets"); title(ax, "D", "Hygiene-only stage"); src("FigureS6", "D", s1)
ax = axs[1, 1]
sc = ax.scatter(b15.a, b15.ratio_treated_naive, s=6, c=b15.K, cmap="viridis", alpha=0.8)
for e_, ls in ((0.03, "-"), (0.01, "--")):
    g = A3W[(A3W.eps == e_) & (A3W.rec_memory_units == 0) & (A3W.alpha >= 0)].sort_values("alpha")
    ax.plot(g.alpha, g.ratio, ls, color="k", lw=1.0, label=f"third set, ε = {e_}")
cb = fig.colorbar(sc, ax=ax, fraction=0.05, pad=0.02); cb.set_label("K", fontsize=6); cb.ax.tick_params(labelsize=5.5)
ax.axhline(1, color="k", lw=0.6); ax.set_xlabel("adaptation strength α"); ax.set_ylabel("treated / naive")
title(ax, "E", "Window grows with α"); ax.legend(loc="lower left")
src("FigureS6", "E", b15)
ax = axs[1, 2]
b3 = r14("14B_tipping_window", "B3_recolonization_check.csv")
hill_ref = {0.03: float(b3[(b3.r == 1.0) & (b3.host == "eps=0.03")].ratio.iloc[0]), 0.01: float(b3[(b3.r == 1.0) & (b3.host == "eps=0.01")].ratio.iloc[0])}
hill_third = {e_: float(A3W[(A3W.alpha == 0.5) & (A3W.eps == e_) & (A3W.rec_memory_units == 0)].ratio.iloc[0]) for e_ in (0.03, 0.01)}
labels, hv, wv = [], [], []
for s_, hd in (("reference", hill_ref), ("third", hill_third)):
    for e_ in (0.03, 0.01):
        labels.append(f"{s_}\nε = {e_}"); hv.append(hd[e_])
        wv.append(float(G15[(G15.set == s_) & (G15.eps == e_)].ratio.iloc[0]))
xx = np.arange(len(labels))
ax.bar(xx - 0.18, hv, 0.36, color="#9aa9c9", label="Hill"); ax.bar(xx + 0.18, wv, 0.36, color=TREAT, label="Weibull")
ax.axhline(1, color="k", lw=0.6); ax.set_xticks(xx); ax.set_xticklabels(labels, fontsize=5.8); ax.set_ylim(0, 1.1)
ax.set_ylabel("treated / naive"); title(ax, "F", "Response functions"); ax.legend(loc="upper left"); ax.grid(axis="x")
src("FigureS6", "F", pd.DataFrame(dict(case=labels, hill=hv, weibull=wv)))
finish(fig, "FigureS6")

### Figure S11

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(WIDTH, 9.2))
ax = axs[0, 0]
for h in ("eps=0.03", "eps=0.01"):
    for hist, ls in (("treated_protocol", "-"), ("treated_reset", "--")):
        g = B2[(B2.host == h) & (B2.history == hist)].sort_values("eps_T_rec")
        ax.plot(g.eps_T_rec, g.critical, ls, color=host_color(h), label=f"ε = {h.split('=')[1]}, {'defined therapy' if hist == 'treated_protocol' else 'state reset'}")
    ax.axhline(float(g.critical_naive.iloc[0]), ls=":", color=host_color(h), lw=0.8)
ax.set_xlabel("recovery (memory times)"); ax.set_ylabel("critical challenge (2/ε)"); title(ax, "A", "Definition of therapy"); ax.legend(loc="lower right")
src("FigureS11", "A", B2)
ax = axs[0, 1]
for kind, ls, lab in (("treated_protocol", "-", "defined therapy"), ("treated_reset", "--", "state reset")):
    g = B4[(B4.kind == kind) & (B4.host != "instant")].copy(); g["eps"] = [float(h.split("=")[1]) for h in g.host]
    g = g.sort_values("eps"); ax.plot(g.eps, g.ceiling, "o" + ls, color=TREAT, label=lab)
ax.axhline(U_UP_REF, ls=":", color="k", lw=0.8); ax.axhline(0.234, ls="--", color="k", lw=0.6)
ax.set_xscale("log"); ax.set_xlabel("memory rate ε"); ax.set_ylabel("relapse ceiling"); title(ax, "B", "Relapse ceiling, both definitions"); ax.legend(loc="lower right")
src("FigureS11", "B", B4)
ax = axs[1, 0]
fr = [0.5, 0.6, 0.7, 0.8, 0.9]
for h in ("eps=0.1", "eps=0.03", "eps=0.01"):
    g = B2H[(B2H.host == h) & (B2H.history == "treated_protocol")]
    ax.plot(fr, [float(g[f"window_at_{f_}_naive_memory_units"].iloc[0]) for f_ in fr], "o-", color=host_color(h), label=f"ε = {h.split('=')[1]}")
ax.set_xlabel("relevant challenge (fraction of naive threshold)"); ax.set_ylabel("window (memory times)"); title(ax, "C", "Length of the window"); ax.legend(loc="upper left")
src("FigureS11", "C", B2H)
ax = axs[1, 1]
for h in ("eps=0.1", "eps=0.03", "eps=0.01"):
    g = B5[B5.host == h]; ax.plot(g.plaque_control_days, g.critical_treated, "o-", color=host_color(h), label=f"ε = {h.split('=')[1]}")
    dur = 2 / float(h.split("=")[1]); n_ = B1[(B1.host == h) & (B1.history == "naive")]
    n_ = n_.iloc[(n_.duration - dur).abs().argsort()[:1]]; ax.axhline(float(n_.critical.iloc[0]), ls=":", color=host_color(h), lw=0.8)
ax.set_xlabel("perfect plaque control after debridement (days)"); ax.set_ylabel("critical challenge (2/ε)"); title(ax, "D", "Plaque control after debridement"); ax.legend(loc="lower right")
src("FigureS11", "D", B5)
for ax, setname, r_, letter, ttl in ((axs[2, 0], "reference", 0.03, "E", "Calibrated rechallenge, r = 0.03"),
                                     (axs[2, 1], "third", 0.03, "F", "Calibrated rechallenge, third set, r = 0.03")):
    c = C15[(C15.set == setname) & (C15.r == r_) & (C15.challenge == "own U*")]
    nv = c[(c.history == "naive") & (c.host == "eps=0.01")]; ax.plot(nv.days, nv.peak_H, "--", color=NAIVE, label="naive")
    for h in ("instant", "eps=0.03", "eps=0.01"):
        g = c[(c.history == "treated") & (c.host == h)]
        ax.plot(g.days, g.peak_H, "o-", color=host_color(h), label=("treated, instant host" if h == "instant" else f"treated, ε = {h.split('=')[1]}"))
        gp = g[g.persistent]; ax.plot(gp.days, gp.peak_H, "o", mfc="white", mec=host_color(h), ms=5, zorder=5)
    ax.set_ylim(0, 1.05); ax.set_xlabel("days since therapy"); ax.set_ylabel("peak H"); title(ax, letter, ttl); ax.legend(loc="upper right")
    src("FigureS11", letter, c)
for ax, col, letter, ttl, yl in ((axs[3, 0], "U_safe", "G", "Largest challenge that never converts", "plateau U"),
                                 (axs[3, 1], "excess_at_U_safe", "H", "Excess response at that challenge", "treated − naive peak H")):
    for h in ("eps=0.03", "eps=0.01"):
        for dd, ls in ((7, ":"), (14, "--"), (21, "-")):
            g = E15M[(E15M.host == h) & (E15M.duration_days == dd)].sort_values("days")
            ax.plot(g.days, g[col], ls, marker="o", ms=2.5, color=host_color(h))
            cap = g[g.lowest_converting_U.isna()]   # no conversion up to the top of the grid
            ax.plot(cap.days, cap[col], "o", mfc="white", mec=host_color(h), ms=3.5, zorder=5)
    ax.set_xlabel("days since therapy"); ax.set_ylabel(yl); title(ax, letter, ttl)
axs[3, 0].legend(handles=[Line2D([], [], color=BLUE[0.03], label="ε = 0.03"), Line2D([], [], color=BLUE[0.01], label="ε = 0.01"),
                          Line2D([], [], color="k", ls=":", label="7 days"), Line2D([], [], color="k", ls="--", label="14 days"),
                          Line2D([], [], color="k", ls="-", label="21 days")], loc="lower right")
src("FigureS11", "G_H", E15M)
finish(fig, "FigureS11")

### Figure S12

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(WIDTH, 4.6))
for ax, ub, letter in ((axs[0, 0], 0.1, "A"), (axs[0, 1], 0.2, "B")):
    x = RC[(RC.baseline == ub) & (RC.U == 0.4) & (~RC.no_challenge) & RC.history.isin(["treated_D0", "treated_Dres"])]
    for h in ("instant", "eps=0.03", "eps=0.01"):
        for hist, ls in (("treated_D0", "-"), ("treated_Dres", "--")):
            g = x[(x.host == h) & (x.history == hist)].sort_values("recovery_days")
            ax.plot(g.recovery_days, g.peak_H, ls, marker="o", ms=2.5, color=host_color(h))
            gp = g[g.persistent]; ax.plot(gp.recovery_days, gp.peak_H, "o", mfc="white", mec=host_color(h), ms=4.5, zorder=5)
    ax.set_ylim(0, 1.05); ax.set_xlabel("days since therapy"); ax.set_ylabel("peak H (plateau 0.40)")
    title(ax, letter, f"Residual attachment loss, baseline {ub:.2f}"); src("FigureS12", letter, x)
axs[0, 0].legend(handles=[Line2D([], [], color=INST, label="instant host"), Line2D([], [], color=BLUE[0.03], label="ε = 0.03"),
                          Line2D([], [], color=BLUE[0.01], label="ε = 0.01"), Line2D([], [], color="k", label="D = 0"),
                          Line2D([], [], color="k", ls="--", label="D = 0.10")], loc="center right")
for ax, ub, letter in ((axs[0, 2], 0.1, "C"), (axs[1, 0], 0.2, "D")):
    for h in ("instant", "eps=0.03", "eps=0.01"):
        for hist, ls in (("treated_D0", "-"), ("treated_Dres", "--")):
            g = LP[(LP.baseline == ub) & (LP.host == h) & (LP.history == hist)].set_index("recovery_days").reindex(days_all)
            rw = RW[(RW.baseline == ub) & (RW.host == h) & (RW.history == hist)]
            relapse = [d for d in days_all if len(rw) and bool(rw[str(float(d))].iloc[0])]
            g.loc[relapse, "lowest_U_persistent"] = np.nan
            ax.plot(g.index, g.lowest_U_persistent, ls, marker="o", ms=2.5, color=host_color(h))
            miss = g[g.lowest_U_persistent.isna()].index
            ax.plot([d for d in miss if d not in relapse], np.full(len([d for d in miss if d not in relapse]), 0.72), "^", mfc="white", mec=host_color(h), ms=3.5)
            ax.plot(relapse, np.full(len(relapse), ub), "x", color=host_color(h), ms=4)
    ax.set_ylim(0.05, 0.75); ax.set_xlabel("days since therapy"); ax.set_ylabel("smallest converting plateau")
    title(ax, letter, f"Conversion by the challenge, {ub:.2f}"); src("FigureS12", letter, LP[LP.baseline == ub])
ax = axs[1, 1]
for e_ in (0.03, 0.01):
    g = D15[(D15.eps == e_) & (D15.U == 0.4)].sort_values("days")
    ax.plot(g.days, g.H_rest_treated, "o-", color=BLUE[e_], label=f"treated, ε = {e_}")
ax.plot(g.days, g.H_rest_naive, "--", color=NAIVE, label="naive")
ax.set_xlabel("days since therapy"); ax.set_ylabel("resting H at plaque 0.20"); title(ax, "E", "Resting inflammation after therapy"); ax.legend(loc="upper right")
src("FigureS12", "E", D15)

ax = axs[1, 2]
g = D15[D15.eps == 0.01]
Us = sorted(g.U.unique())
for U_, ls in zip(Us, ("-", "--")):
    gg = g[g.U == U_].sort_values("days")
    ax.plot(gg.days, gg.peak_treated, ls, color=TREAT)
    ax.plot(gg.days, gg.peak_naive_matched_same_increment, ls, color=NAIVE)
    gp = gg[gg.persistent_treated]; ax.plot(gp.days, gp.peak_treated, "o", mfc="white", mec=TREAT, ms=4.5, zorder=5)
ax.set_ylim(0, 1.3); ax.set_xlabel("days since therapy"); ax.set_ylabel("peak H"); title(ax, "F", "Matched resting H (ε = 0.01)")
ax.legend(handles=[Line2D([], [], color=TREAT, label="treated"), Line2D([], [], color=NAIVE, label="naive, matched"),
                   Line2D([], [], color="k", label=f"plateau {Us[0]:.2f}"), Line2D([], [], color="k", ls="--", label=f"plateau {Us[1]:.2f}")],
          loc="upper right", ncol=2); src("FigureS12", "F", g)
finish(fig, "FigureS12")

### Figure S13

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(WIDTH, 4.6))
ax = axs[0, 0]
f2 = FOLDS2.sort_values("alpha")
ax.fill_between(f2.alpha, np.maximum(f2.u_down, 0), f2.u_up, color="#c8c2f0", alpha=0.6, label="two stable states")
ax.plot(f2.alpha, f2.u_up, color=TREAT, lw=1.0); ax.plot(f2.alpha, np.maximum(f2.u_down, 0), "--", color=TREAT, lw=1.0)
ax.axhline(UB2, ls=":", color="k", lw=0.8)
ax.axvspan(A_FAIL - 0.0125, 0.52, facecolor="#eeeeee", edgecolor="#bbbbbb", hatch="///", lw=0, zorder=0)
ax.text(A_FAIL - 0.005, 0.455, "therapy cannot\nrestore the site", fontsize=5.5, va="top")
cc = C2[(C2.host == "eps=0.03") & C2.relapse_ceiling.notna() & (C2.alpha < A_FAIL)]
ax.plot(cc.alpha, cc.relapse_ceiling, "D", color="k", ms=4, label="relapse ceiling (defined therapy)")
ax.set_xlim(-0.52, 0.52); ax.set_ylim(0, 0.47); ax.set_xlabel("adaptation strength α"); ax.set_ylabel("plaque u")
title(ax, "A", "Two stable states, second set"); ax.legend(loc="lower left")
src("FigureS13", "A", f2.merge(cc, on="alpha", how="left"))
ax = axs[0, 1]
pt = r11("periodic_thresholds_by_sign.csv")
for h, col, lab in (("eps=0.03", BLUE[0.03], "memory (ε = 0.03)"), ("instant", INST, "instant host")):
    g = pt[pt.host == h].sort_values("a"); ax.plot(g.a, g.threshold_over_fold, "o-", color=col, label=lab)
    nev = g[g.threshold_over_fold.isna()]
    ax.plot(nev.a, np.full(len(nev), 1.95), "^", mfc="white", mec=col, ms=4.5)
    for a_ in nev.a: ax.annotate("never entrapped\n(peak ≤ 0.9)", (a_, 1.95), (a_ + 0.05, 1.70), fontsize=5.5)
ax.set_xlabel("adaptation strength α"); ax.set_ylabel("tolerated peak / static fold"); title(ax, "B", "Periodic plaque (T = 60)"); ax.legend(loc="upper right")
src("FigureS13", "B", pt)
ax = axs[1, 0]
dv = r11("debridement_by_sign.csv")
cols = {-0.5: GING, -0.25: "#7fd3b4", 0.25: "#a093e8", 0.5: TREAT, 0.0: "#888888"}
for a in sorted(dv.a.unique()):
    for h, ls in (("eps=0.03", "-"), ("instant", "--")):
        g = dv[(dv.a == a) & (dv.host == h)].sort_values("visit")
        if len(g): ax.plot(g.visit, g.relapse_free_fraction, ls, color=cols.get(a, "k"), lw=1.0, label=(f"α = {a}" if h == "eps=0.03" else None))
ax.set_xlabel("visit"); ax.set_ylabel("inflammation-free fraction"); title(ax, "C", "Debridement at every visit"); ax.legend(loc="upper right", ncol=2)
src("FigureS13", "C", dv)
ax = axs[1, 1]
for h, col in (("eps=0.03", BLUE[0.03]), ("instant", INST)):
    g = TR2[TR2.host == h]; ax.plot(g.t, g.H, color=col, label=f"H, {'memory' if h != 'instant' else 'instant host'}")
    if h != "instant": ax.plot(g.t, g.A, ":", color=col, label="A, memory")
ax.set_xlabel("time (model units), last two of 40 intervals"); ax.set_ylabel("H, A"); title(ax, "D", "Tolerance carried across resets (α = −0.5)"); ax.legend(loc="center right")
src("FigureS13", "D", TR2)
finish(fig, "FigureS13")

### Figure S15, new panels E–F (to be placed under the existing panels A–D)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(WIDTH, 2.3))
ax = axs[0]
g0 = F2[F2.r == 0.03]
for h in ("instant", "eps=0.01", "eps=0.003"):
    for pk, ls in ((0.6, "-"), (0.75, "--")):
        g = g0[(g0.host == h) & (g0.peak == pk)].sort_values("visit")
        ax.plot(g.visit, g.free_fraction, ls, color=host_color(h), lw=1.0)
ax.legend(handles=[Line2D([], [], color=INST, label="instant host"), Line2D([], [], color=BLUE[0.01], label="ε = 0.01"),
                   Line2D([], [], color=BLUE[0.003], label="ε = 0.003"), Line2D([], [], color="k", label="peak 0.60"),
                   Line2D([], [], color="k", ls="--", label="peak 0.75")], loc="upper center", ncol=3)
ax.set_ylim(0, 1.32); ax.set_xlabel("visit"); ax.set_ylabel("inflammation-free fraction"); title(ax, "E", "Debridement at every recall, r = 0.03")
src("FigureS15", "E", g0)
ax = axs[1]
g1 = F1[(F1.r == 0.03) & (F1.maintenance_peak == 0.3)]
for h in ("instant", "eps=0.03", "eps=0.01"):
    for arm, ls in (("single", "--"), ("repeated", "-")):
        g = g1[(g1.host == h) & (g1.arm == arm)].sort_values("D0")
        ax.plot(g.D0, g.inflammation_free_fraction, ls, marker="o", ms=2.5, color=host_color(h))
ax.legend(handles=[Line2D([], [], color=INST, label="instant host"), Line2D([], [], color=BLUE[0.03], label="ε = 0.03"),
                   Line2D([], [], color=BLUE[0.01], label="ε = 0.01"), Line2D([], [], color="k", ls="--", label="single"),
                   Line2D([], [], color="k", label="every recall")], loc="upper center", ncol=3)
ax.set_ylim(0, 1.32); ax.set_xlabel("attachment loss at therapy D0"); ax.set_ylabel("inflammation-free fraction")
title(ax, "F", "Single vs repeated instrumentation (peak 0.30)"); src("FigureS15", "F", g1)
finish(fig, "FigureS15_EF")

## Numbers for the text, checks and export

In [ ]:
z = W2[W2.eps_T_rec == 0].set_index("alpha")
summary = {
    "second set, defined therapy, treated/naive at alpha=-0.25": z.loc[-0.25, "ratio_treated"],
    "second set, state reset, treated/naive at alpha=-0.25": z.loc[-0.25, "ratio_reset"],
    "second set, defined therapy, treated/naive at alpha=0.25": z.loc[0.25, "ratio_treated"],
    "second set, state reset, treated/naive at alpha=0.25": z.loc[0.25, "ratio_reset"],
    "second set, therapy restores at alpha=0.5": bool(z.loc[0.5, "therapy_restores"]),
    "dose-dependent, gingivitis ratio at recovery 0": float(D2[(D2.history == "gingivitis") & (D2.eps_T_rec == 0)].ratio.iloc[0]),
    "dose-dependent, treated ratio at recovery 0": float(D2[(D2.history == "treated_protocol") & (D2.eps_T_rec == 0)].ratio.iloc[0]),
    "relapse ceiling / fold at alpha=0.25 (eps=0.03)": C2[(C2.alpha == 0.25) & (C2.host == "eps=0.03")].ceiling_over_fold.iloc[0],
    "15B evaluable sets": int((B15.category == "evaluable").sum()),
    "15B treated/naive median": float(b15.ratio_treated_naive.median()),
    "14E treated/naive median": float(e14.ratio_treated_naive.median()),
}
S = pd.DataFrame(dict(item=list(summary), value=list(summary.values())))
S.to_csv(os.path.join(OUT, "numbers_for_text.csv"), index=False); print(S.to_string())
saved = [pd.read_csv(f_) for f_ in sorted(glob.glob(os.path.join(OUT, "*", "checks_*.csv")))]
unsaved = pd.DataFrame([c_ for c_ in CHECKS if not c_.get("saved")], columns=["module", "check", "value", "passed"])
pd.concat(saved + [unsaved], ignore_index=True).to_csv(os.path.join(OUT, "all_checks.csv"), index=False)
open(os.path.join(OUT, "README.txt"), "w").write("""Experiment 16 - Revision figures
16A_second_set_defined_therapy: sign of adaptation on the second set with the defined therapy (window_by_alpha.csv:
  treated/naive and state-reset/naive critical challenge vs recovery; instant_host.csv; dose_dependent.csv;
  relapse_ceiling.csv; folds_by_alpha.csv; traces_alpha_m05.csv).
figures/: Figure3, Figure4, FigureS6, FigureS11, FigureS12, FigureS13 (complete) and FigureS15_EF (new panels E-F),
  each as PDF, 600-dpi PNG and per-panel CSV.
numbers_for_text.csv: values quoted in the revised text. Inputs in _inputs/ (Experiments 11, 14) and the Experiment 15 export.
""")
shutil.rmtree(INPUTS, ignore_errors=True)
zip_path = shutil.make_archive(OUT, "zip", os.path.dirname(os.path.abspath(OUT)), os.path.basename(OUT))
print("wrote", zip_path)
if IN_COLAB:
    try:
        from google.colab import files; files.download(zip_path)
    except Exception as e:
        print("download did not start; the zip is in Drive:", e)